# Claims desk · Build a supervisor with LangGraph

**Factored technical workshop · Multi-agent workflows**

You will build the core of the claims desk from the live demo, one piece at a time:

1. **State**: the shared, typed memory every node reads and writes.
2. **A workflow**: two nodes in a fixed order.
3. **A supervisor**: an LLM that picks the next step, inside rules written in code.
4. **A human in the loop**: the graph pauses, a person decides, the graph resumes.
5. **Your turn**: predict, then run.

**Bonus:** a document intake with its own human check.

After each part you will **draw the graph** and **watch it run**. The data is synthetic and in US dollars. Nothing here approves or pays a claim.

---
## Before you start: your API key

We share one OpenRouter key for today through a secure link from a password manager (Bitwarden Send or similar). Open the link the facilitators share to reveal the key. It stops working after the session.

1. In Colab, open **Secrets** (key icon, left sidebar).
2. Add a secret named `OPENROUTER_API_KEY`, paste the key, turn on **notebook access**.
3. Run the notebook from the top.

Do not paste the key into a code cell or share it.

No key, or it stopped working? Ask a facilitator.

---
## Setup

On Colab, the first cell installs the libraries. Locally, run `uv sync` inside `notebooks/` and pick that environment as the kernel.

The setup cell reads the key from Colab Secrets (locally, from `OPENROUTER_API_KEY` in your environment or `.env`). If it finds none, it asks for it in a hidden prompt. It also defines two helpers:

- `call_llm(system_prompt, user_message, output_schema)` calls the model and returns its answer as a validated Pydantic object: **structured output**.
- `show_graph(graph)` draws a compiled graph.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "langgraph>=1.0,<2" "openai>=2,<3" "pydantic>=2,<3"

In [ ]:
import json
import operator
import os
import re
import sys
import uuid
from datetime import date
from pathlib import Path
from typing import Annotated, Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from openai import OpenAI
from pydantic import BaseModel, ConfigDict

MODEL = "qwen/qwen3.5-flash-02-23"


def load_api_key() -> str:
    """Read OPENROUTER_API_KEY from the environment, a local .env, Colab Secrets, or a prompt."""
    key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    for path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
        if not key and path.is_file():
            for line in path.read_text(encoding="utf-8").splitlines():
                name, _, value = line.partition("=")
                if name.strip() == "OPENROUTER_API_KEY":
                    key = value.strip().strip('"').strip("'")
    if not key and "google.colab" in sys.modules:
        from google.colab import userdata

        try:
            key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
        except Exception:  # no secret yet, or notebook access not granted
            key = ""
    if not key:
        from getpass import getpass

        key = getpass("OPENROUTER_API_KEY: ").strip()
    if not key:
        raise RuntimeError("An OpenRouter key is required. See the Setup notes above.")
    return key


client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_api_key(), timeout=60.0)


def call_llm(system_prompt: str, user_message: str, output_schema: type[BaseModel]) -> BaseModel:
    """Call the model and return its answer as a validated `output_schema` object."""
    keys = ", ".join(output_schema.model_fields)
    messages = [
        {"role": "system", "content": f"{system_prompt} Reply with one JSON object with these keys: {keys}."},
        {"role": "user", "content": user_message},
    ]
    for _attempt in range(2):
        completion = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=messages,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": output_schema.__name__,
                    "strict": True,
                    "schema": output_schema.model_json_schema(),
                },
            },
            # Only use providers that enforce the schema; skip the model's thinking step.
            extra_body={"provider": {"require_parameters": True}, "reasoning": {"enabled": False}},
        )
        reply = completion.choices[0].message.content or ""
        json_text = reply.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
        try:
            return output_schema.model_validate_json(json_text)
        except ValueError as error:  # show the model its mistake once, then retry
            messages += [
                {"role": "assistant", "content": reply},
                {"role": "user", "content": f"Invalid answer: {error}. Return only the JSON object."},
            ]
    raise RuntimeError(f"The model did not return a valid {output_schema.__name__}.")


def show_graph(graph) -> None:
    """Draw a compiled graph; print its Mermaid text if the image service is unreachable."""
    try:
        from IPython.display import Image, display

        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())


print("Ready:", MODEL)

---
## Part 1 · State: the shared memory

Every node in a LangGraph graph reads the **state** and returns **only the keys it changes**. LangGraph merges those updates for you.

We type the state so it documents itself. Each specialist writes **the field with its own name** (`coverage` writes `coverage`), with a fixed shape, and the field stays `None` until that specialist has run.

`audit_trail` uses a **reducer**, `operator.add`: updates are **appended** instead of replacing the list. That records every routing decision for free.

In [ ]:
CLAIMS = {
    "AUTO-202": {
        "vehicle": "Toyota Corolla 2019",
        "description": "Localized side door damage; inspection complete.",
        "repair_estimate_usd": 18_000,
        "reference_value_usd": 80_000,
        "total_loss_threshold": 0.80,
    },
    "AUTO-101": {
        "vehicle": "Honda Civic 2020",
        "description": "Front impact; possible hidden structural damage.",
        "repair_estimate_usd": 82_000,
        "reference_value_usd": 100_000,
        "total_loss_threshold": 0.75,
    },
}


class CoverageFinding(TypedDict):
    covered: bool
    reference_value_usd: int
    total_loss_threshold: float


class DamageFinding(TypedDict):
    repair_estimate_usd: int


class AdjusterReview(TypedDict):
    status: Literal["pending", "total_loss", "repair"]


class AuditEntry(TypedDict):
    next: str
    reason: str


class ClaimState(TypedDict):
    claim_id: str
    coverage: CoverageFinding | None
    damage: DamageFinding | None
    adjuster: AdjusterReview | None
    audit_trail: Annotated[list[AuditEntry], operator.add]  # reducer: append, don't replace


def new_claim(claim_id: str) -> ClaimState:
    return {"claim_id": claim_id, "coverage": None, "damage": None, "adjuster": None, "audit_trail": []}


def loss_ratio(state: ClaimState) -> float:
    return state["damage"]["repair_estimate_usd"] / state["coverage"]["reference_value_usd"]


def format_usd(amount: float) -> str:
    return f"${amount:,.0f}"


def format_percent(ratio: float) -> str:
    return f"{ratio:.1%}".replace(".0%", "%")


new_claim("AUTO-202")

---
## Part 2 · A first graph: a workflow

A **node** is a plain function: state in, update out. **Edges** fix the order. No model is involved yet, so this is a **workflow**: the same steps every time.

You assemble a graph with a `builder`, then `compile()` it into something you can run.

In the full desk, coverage and damage are small agents with their own tools (policy lookup, photos, price table). Here they read canned data, so we can focus on the graph itself.

In [ ]:
def coverage(state: ClaimState) -> dict:
    claim = CLAIMS[state["claim_id"]]
    finding = CoverageFinding(
        covered=True,
        reference_value_usd=claim["reference_value_usd"],
        total_loss_threshold=claim["total_loss_threshold"],
    )
    return {"coverage": finding}


def damage(state: ClaimState) -> dict:
    claim = CLAIMS[state["claim_id"]]
    return {"damage": DamageFinding(repair_estimate_usd=claim["repair_estimate_usd"])}


builder = StateGraph(ClaimState)
builder.add_node("coverage", coverage)
builder.add_node("damage", damage)
builder.add_edge(START, "coverage")
builder.add_edge("coverage", "damage")
builder.add_edge("damage", END)
workflow = builder.compile()

show_graph(workflow)

In [ ]:
final_state = workflow.invoke(new_claim("AUTO-202"))

print("Coverage:", final_state["coverage"])
print("Damage:  ", final_state["damage"])
print("Loss ratio:", format_percent(loss_ratio(final_state)))

**What just happened:** `invoke` ran `START → coverage → damage → END`. Each node returned a single key, and LangGraph merged them into the final state.

---
## Part 3 · The supervisor: an LLM picks the next step

Now a **supervisor** decides what runs next. Two ideas keep it safe:

- **Code decides what is allowed; the model picks among the allowed steps.** Here: coverage and damage first, in either order. Then the adjuster when the ratio meets the threshold, otherwise finish.
- **Routing with `Command`.** The supervisor returns `Command(goto=..., update=...)`: where to go next, plus what to record. The type hint `Command[Literal[...]]` declares every possible destination, so LangGraph can still draw them all: in the drawing, **dotted arrows** are choices and **solid arrows** always run.

The model's answer is a Pydantic model, `RouteDecision`, so it is validated before it touches the state. The state itself stays plain typed dicts.

For now the adjuster only **flags** the case for a person. In Part 4 it will really stop and wait.

In [ ]:
class RouteDecision(BaseModel):
    """The supervisor's answer. Pydantic validates what the model returns."""

    model_config = ConfigDict(extra="forbid")
    next: Literal["coverage", "damage", "adjuster", "FINISH"]
    reason: str


SUPERVISOR_PROMPT = (
    "You supervise an insurance claims investigation. "
    "Pick the next step from allowed_next only, and give a one-sentence reason. "
    "Coverage and damage come first. The adjuster is a person, needed when the loss ratio "
    "meets the policy's total-loss threshold. Choose FINISH when nothing is left to do. "
    "Never approve or deny a claim."
)


def allowed_next_steps(state: ClaimState) -> list[str]:
    missing = [name for name in ("coverage", "damage") if state[name] is None]
    if missing:
        return missing
    if state["adjuster"] is not None:
        return ["FINISH"]
    if loss_ratio(state) >= state["coverage"]["total_loss_threshold"]:
        return ["adjuster"]
    return ["FINISH"]


def supervisor(state: ClaimState) -> Command[Literal["coverage", "damage", "adjuster", "__end__"]]:
    allowed_next = allowed_next_steps(state)
    claim_summary = {
        "claim": CLAIMS[state["claim_id"]]["description"],
        "coverage": state["coverage"],
        "damage": state["damage"],
        "adjuster": state["adjuster"],
        "loss_ratio": round(loss_ratio(state), 3) if state["coverage"] and state["damage"] else None,
        "allowed_next": allowed_next,
    }
    decision = call_llm(SUPERVISOR_PROMPT, json.dumps(claim_summary), output_schema=RouteDecision)
    if decision.next not in allowed_next:  # the guard: code has the last word
        decision = RouteDecision(next=allowed_next[0], reason=f"Guard: {decision.next} is not allowed now.")
    next_node = END if decision.next == "FINISH" else decision.next
    return Command(goto=next_node, update={"audit_trail": [decision.model_dump()]})

In [ ]:
def flag_for_adjuster(state: ClaimState) -> dict:
    return {"adjuster": AdjusterReview(status="pending")}


def build_desk(adjuster_node, checkpointer=None):
    builder = StateGraph(ClaimState)
    builder.add_node("supervisor", supervisor)
    builder.add_node("coverage", coverage)
    builder.add_node("damage", damage)
    builder.add_node("adjuster", adjuster_node)
    builder.add_edge(START, "supervisor")
    for specialist in ("coverage", "damage", "adjuster"):
        builder.add_edge(specialist, "supervisor")  # every specialist reports back
    return builder.compile(checkpointer=checkpointer)


def watch(graph, graph_input, config=None) -> None:
    """Run the graph on a new state or a Command, printing each node's update as it happens."""
    for step in graph.stream(graph_input, config, stream_mode="updates"):
        for node, update in step.items():
            if node == "__interrupt__":
                print("⏸  paused for a person:", update[0].value)
            elif node == "supervisor":
                for entry in update["audit_trail"]:
                    print(f"▶ supervisor → {entry['next']}: {entry['reason']}")
            else:
                print(f"▶ {node}: {update}")


desk = build_desk(flag_for_adjuster)
show_graph(desk)

In [ ]:
watch(desk, new_claim("AUTO-202"))

**What just happened:** each line is one step. The supervisor sent the case to coverage and damage, then finished at 22.5%. No person was needed.

In [ ]:
watch(desk, new_claim("AUTO-101"))

**The problem:** AUTO-101 is at 82%, so the supervisor sent it to the adjuster. But the graph just wrote `pending` and finished. Nobody actually decided. **A flag is not a checkpoint.**

---
## Part 4 · Human in the loop: pause, persist, resume

Three pieces make a real checkpoint:

1. A **checkpointer** saves the state after every step, under a `thread_id`.
2. `interrupt(review_request)` stops the graph inside a node and hands the request to a person.
3. `Command(resume=...)` continues from that point. The node runs again from its start, and this time `interrupt()` returns the person's response, so put side effects after it.

In [ ]:
def adjuster(state: ClaimState) -> dict:
    review_request = {
        "claim_id": state["claim_id"],
        "repair_estimate": format_usd(state["damage"]["repair_estimate_usd"]),
        "reference_value": format_usd(state["coverage"]["reference_value_usd"]),
        "loss_ratio": format_percent(loss_ratio(state)),
        "threshold": format_percent(state["coverage"]["total_loss_threshold"]),
        "question": "Review as total_loss or repair?",
    }
    human_response = interrupt(review_request)
    if human_response.get("choice") not in ("total_loss", "repair"):
        raise ValueError('Resume with {"choice": "total_loss"} or {"choice": "repair"}.')
    return {"adjuster": AdjusterReview(status=human_response["choice"])}


desk = build_desk(adjuster, checkpointer=InMemorySaver())
show_graph(desk)

In [ ]:
config = {"configurable": {"thread_id": f"AUTO-101-{uuid.uuid4().hex[:6]}"}}
watch(desk, new_claim("AUTO-101"), config)

In [ ]:
snapshot = desk.get_state(config)
print("Waiting in:", snapshot.next)
print("Review request:", snapshot.tasks[0].interrupts[0].value)

**What just happened:** the graph is not running anymore. Its state is saved in the checkpointer, and `next` shows it is waiting inside the adjuster node. In a real app this pause could last hours, and the person would answer from a screen.

In [ ]:
watch(desk, Command(resume={"choice": "total_loss"}), config)

final_state = desk.get_state(config).values
print("Adjuster:", final_state["adjuster"]["status"], "· ratio still", format_percent(loss_ratio(final_state)))

**What just happened:** the person's choice is recorded on the case; the numbers did not move. To record the other choice, run the AUTO-101 cell again (it starts a new thread), then resume with `"repair"`.

---
## Part 5 · Your turn

1. Change the repair estimate below. **Predict first:** will it pause for a person? The reference value is $80,000 and the threshold is 80%.
2. Run it, then read the audit trail that the reducer collected.
3. If it paused, decide for the adjuster: `watch(desk, Command(resume={"choice": "repair"}), config)`.

In [ ]:
CLAIMS["MY-CLAIM"] = {**CLAIMS["AUTO-202"], "repair_estimate_usd": 70_000}  # ← change me

config = {"configurable": {"thread_id": f"MY-CLAIM-{uuid.uuid4().hex[:6]}"}}
watch(desk, new_claim("MY-CLAIM"), config)

print("\nAudit trail:")
for entry in desk.get_state(config).values["audit_trail"]:
    print(f"- {entry['next']}: {entry['reason']}")

---
## Bonus · A document with a human check (optional)

In the demo, the desk read a PDF and paused when a field was unreadable. Here is the same idea on a short claim text, as its own small graph:

`extract` (the model reads the document into fields) → `validate` (code checks them) → if a field is invalid, `review` (a person fixes it) → back to `validate`.

This graph branches with a **conditional edge**, the other way to route in LangGraph: a routing function, `needs_review`, reads the state and returns the name of the next node.

In [ ]:
CLAIM_TEXT = (
    "Claim AUTO-404 · Policy POL-7781\n"
    "Vehicle: Honda Civic 2020 · Plate: ABC1D23\n"
    "Incident date: 2026-03-14\n"
    "Repair estimate: $8?,000 (scan smudged)"
)


class ClaimFields(BaseModel):
    """The fields the model extracts. Pydantic validates what it returns."""

    model_config = ConfigDict(extra="forbid")
    claim_id: str
    plate: str
    incident_date: str
    repair_estimate_usd: int | None


class IntakeState(TypedDict):
    document: str
    fields: dict
    invalid_fields: list[str]


EXTRACT_PROMPT = (
    "Extract the claim fields from the document. Dates as YYYY-MM-DD, amounts as whole dollars. "
    "If any character of a value is unreadable, such as a '?', return null for that field. "
    "Never guess."
)


def extract(state: IntakeState) -> dict:
    fields = call_llm(EXTRACT_PROMPT, state["document"], output_schema=ClaimFields)
    return {"fields": fields.model_dump()}


def validate(state: IntakeState) -> dict:
    fields, invalid_fields = state["fields"], []
    if not re.fullmatch(r"[A-Z]{3}\d[A-Z0-9]\d{2}", fields["plate"]):
        invalid_fields.append("plate")
    try:
        date.fromisoformat(fields["incident_date"])
    except ValueError:
        invalid_fields.append("incident_date")
    if not fields["repair_estimate_usd"]:
        invalid_fields.append("repair_estimate_usd")
    return {"invalid_fields": invalid_fields}


def review(state: IntakeState) -> dict:
    corrections = interrupt({"invalid_fields": state["invalid_fields"], "document": state["document"]})
    return {"fields": {**state["fields"], **corrections}}


def needs_review(state: IntakeState) -> Literal["review", "__end__"]:
    return "review" if state["invalid_fields"] else END


builder = StateGraph(IntakeState)
builder.add_node("extract", extract)
builder.add_node("validate", validate)
builder.add_node("review", review)
builder.add_edge(START, "extract")
builder.add_edge("extract", "validate")
builder.add_conditional_edges("validate", needs_review)
builder.add_edge("review", "validate")
intake = builder.compile(checkpointer=InMemorySaver())

show_graph(intake)

In [ ]:
config = {"configurable": {"thread_id": f"AUTO-404-{uuid.uuid4().hex[:6]}"}}
watch(intake, {"document": CLAIM_TEXT, "fields": {}, "invalid_fields": []}, config)

In [ ]:
watch(intake, Command(resume={"repair_estimate_usd": 82_000}), config)
print("Clean claim:", intake.get_state(config).values["fields"])

---
## What you built

- **State** with typed fields, and a **reducer** that keeps an audit trail.
- A **workflow** with fixed edges, then a **supervisor** that routes with `Command` inside rules written in code.
- A **human checkpoint**: checkpointer + `interrupt` + `Command(resume=...)`.
- Bonus: a **conditional edge** and a document check.

## Takeaways

- Draw every path; let the model choose among them.
- Keep the numbers in code. The model decides what happens next, never what things cost.
- Pause where a mistake is expensive. The person's choice is recorded, and the numbers don't move.

The full desk, with PDF reading, tool-calling subagents and the web app, is in this repository: https://github.com/WittmannF/factored-workshop-agentic-workflows